# Model Inference — Best Model from Registry
IEEE-CIS Fraud Detection — Submission Generation

## 0. Setup

In [ ]:
import os, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from kaggle_secrets import UserSecretsClient


DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = 'delibaaa'
REPO_NAME        = 'DELIBA-ML-ASSIGNMENT-2'
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

MLFLOW_URI = f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow'
mlflow.set_tracking_uri(MLFLOW_URI)

print(f'MLflow URI set: {MLFLOW_URI}')

## 1. Load Best Model from Model Registry

In [ ]:
MODEL_NAME    = 'LightGBM_Fraud_Pipeline'
MODEL_VERSION = 'latest'

model_uri = f'models:/{MODEL_NAME}/1'  
print(f'Loading model from: {model_uri}')

loaded_pipeline = mlflow.sklearn.load_model(model_uri)
print(f'Model loaded successfully: {type(loaded_pipeline)}')
print(f'Pipeline steps: {[step[0] for step in loaded_pipeline.steps]}')

## 2. Load & Prepare Test Data (Raw — No Manual Preprocessing)

In [ ]:
DATA_PATH = '/kaggle/input/ieee-fraud-detection'

test_transaction = pd.read_csv(f'{DATA_PATH}/test_transaction.csv')
test_identity    = pd.read_csv(f'{DATA_PATH}/test_identity.csv')
test_raw = test_transaction.merge(test_identity, on='TransactionID', how='left')

print(f'Raw test shape: {test_raw.shape}')
print(f'TransactionIDs: {test_raw["TransactionID"].nunique()}')

## 3. Apply Same Feature Engineering as Training

In [ ]:
def feature_engineering(df):
    df = df.copy()
    df['TransactionHour']    = (df['TransactionDT'] // 3600 % 24).astype(int)
    df['TransactionDay']     = (df['TransactionDT'] // (3600*24)).astype(int)
    df['TransactionWeekDay'] = (df['TransactionDay'] % 7).astype(int)
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

    for col in ['card1', 'card2', 'addr1', 'P_emaildomain', 'R_emaildomain']:
        if col in df.columns:
            df[f'{col}_freq'] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    if 'card1' in df.columns:
        df['card1_amt_mean'] = df.groupby('card1')['TransactionAmt'].transform('mean')
        df['card1_amt_std']  = df.groupby('card1')['TransactionAmt'].transform('std').fillna(0)
        df['card1_amt_diff'] = df['TransactionAmt'] - df['card1_amt_mean']

    if 'P_emaildomain' in df.columns:
        df['P_email_suffix'] = df['P_emaildomain'].str.split('.').str[-1].fillna('unknown')

    return df

test_fe = feature_engineering(test_raw)
print(f'Test after FE: {test_fe.shape}')

## 4. Load Selected Features & Predict

In [ ]:
FEATURES_FILE = 'lgbm_selected_features.json'

with open(FEATURES_FILE, 'r') as f:
    selected_features = json.load(f)

print(f'Using {len(selected_features)} features')

X_test = test_fe[selected_features].copy()
print(f'X_test shape: {X_test.shape}')

print('Running predictions...')
y_pred_proba = loaded_pipeline.predict_proba(X_test)[:, 1]
print(f'Predictions done. Shape: {y_pred_proba.shape}')
print(f'Predicted fraud rate: {y_pred_proba.mean():.4f}')

## 5. Generate Submission File

In [ ]:
submission = pd.DataFrame({
    'TransactionID': test_raw['TransactionID'],
    'isFraud': y_pred_proba
})

print(f'Submission shape: {submission.shape}')
print(submission.head())
print(f'\nPrediction stats:')
print(submission['isFraud'].describe())

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved!')

## 6. Validate Submission Format

In [ ]:
sub = pd.read_csv('submission.csv')

assert list(sub.columns) == ['TransactionID', 'isFraud'], 'Column names wrong!'
assert sub['isFraud'].between(0, 1).all(), 'Probabilities out of [0,1] range!'
assert sub['TransactionID'].nunique() == len(sub), 'Duplicate TransactionIDs!'
assert len(sub) == len(test_raw), f'Row count mismatch! Expected {len(test_raw)}, got {len(sub)}'

print(f'Submission valid!')
print(f'Rows: {len(sub)}')
print(f'Columns: {list(sub.columns)}')
print(f'Probability range: [{sub["isFraud"].min():.6f}, {sub["isFraud"].max():.6f}]')
print(f'Mean predicted fraud probability: {sub["isFraud"].mean():.6f}')